# AIFlow — tạo giọng đọc theo phiên

Chọn **Runtime → Change runtime type → GPU T4**, rồi **Runtime Version → 2026.07** (Python 3.12). Kokoro 0.9.4 yêu cầu Python 3.10–3.12; runtime mặc định Python 3.13 chưa tương thích. Chỉ chạy notebook khi có nội dung đã duyệt. Model không chạy trên máy cá nhân.

Worker v1: Kokoro tiếng Anh, WAV, một tác vụ GPU mỗi lần. TTS hiện chỉ hỗ trợ tiếng Anh. STT tùy chọn hỗ trợ en/vi; chạy ô bật STT bên dưới khi cần.

Colab Pro không có SLA hay tài nguyên vô hạn. Đọc [Colab FAQ](https://research.google.com/colaboratory/faq.html) trước khi dùng tunnel. Nếu tunnel không phù hợp, dùng phần Batch dưới đây. Không dùng keep-alive hoặc lách giới hạn.

Tải `aiflow-audio-worker.zip` từ trang **Colab & giọng đọc** của AIFlow. Không đưa secret vào notebook đã chia sẻ; bật Omit code cell output when saving.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, sys, subprocess, os, secrets, json

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError('Kokoro 0.9.4 cần Python 3.10–3.12. Chọn Runtime → Change runtime type → Runtime Version 2026.07 và GPU T4, rồi chạy lại.')

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Chọn đúng một file aiflow-audio-worker.zip')
archive = zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values()))))
expected = {'audio_worker/app.py', 'audio_worker/requirements.txt'}
if set(archive.namelist()) != expected:
    raise ValueError('Gói worker không đúng định dạng')
worker_root = Path('/content/aiflow-worker')
for name in expected:
    dest = worker_root / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(archive.read(name))
print('Đã nhận worker từ AIFlow của bạn.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.environ['AIFLOW_WORKER_STORAGE'] = '/content/drive/MyDrive/AIFlow/audio-worker'
os.environ['AIFLOW_WORKER_TOKEN'] = secrets.token_urlsafe(48)
subprocess.run(['apt-get', '-qq', 'update'], check=True)
subprocess.run(['apt-get', '-qq', '-y', 'install', 'espeak-ng'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(worker_root / 'audio_worker/requirements.txt')], check=True)

In [ ]:
sys.path.insert(0, str(worker_root))
from audio_worker import app as worker
if worker.ACTIVE:
    raise RuntimeError('Còn tác vụ audio đang chạy. Chờ xong trước khi nạp lại model.')
worker.TOKEN = os.environ['AIFLOW_WORKER_TOKEN']
worker.load_model()
print(json.dumps({'health': worker.health(), 'voices': worker.voices()}, ensure_ascii=False, indent=2))
print('Model sẵn sàng. JSON phía trên là profile cho chế độ Batch, không chứa token.')

## STT tùy chọn
Chỉ bật khi cần nhận dạng phụ đề. Đợi tác vụ hiện tại xong, chạy ô này rồi kiểm tra/lưu lại kết nối trên AIFlow. TTS và STT dùng chung một hàng đợi GPU. CUDA/cuDNN phải tương thích faster-whisper; nếu lỗi nạp, dùng phụ đề theo cảnh.


In [ ]:
ENABLE_STT = False # Đổi thành True khi cần
if ENABLE_STT:
    worker.load_stt_model()
    print("STT ready. Recheck and save the connection in AIFlow.")


## A. Kết nối API tạm

Chỉ chạy phần này nếu cách sử dụng phù hợp với điều khoản Colab hiện hành. Quick Tunnel dành cho phát triển/thử nghiệm, không bảo đảm uptime. Bỏ qua cả hai cell API để sử dụng Batch.

In [ ]:
import threading, urllib.request, stat, uvicorn
if 'server' in globals() and not server.should_exit:
    raise RuntimeError('API đã chạy. Dừng cell cuối trước khi khởi động lại.')
server = uvicorn.Server(uvicorn.Config(worker.app, host='127.0.0.1', port=8000, log_level='warning', access_log=False))
api_thread = threading.Thread(target=server.run, daemon=True)
api_thread.start()
cloudflared = Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(cloudflared.stat().st_mode | stat.S_IXUSR)
tunnel_log = open('/content/aiflow-tunnel.log', 'w')
tunnel = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000'], stdout=tunnel_log, stderr=subprocess.STDOUT)
print('Đã khởi động API và tunnel. Chạy cell tiếp theo để lấy URL.')

In [ ]:
import re, time
url = None
for _ in range(30):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/aiflow-tunnel.log').read_text())
    if match:
        url = match.group(0)
        break
    time.sleep(1)
if not url:
    raise RuntimeError('Chưa có URL tunnel; kiểm tra /content/aiflow-tunnel.log hoặc dùng Batch.')
print('URL:', url)
print('TOKEN:', os.environ['AIFLOW_WORKER_TOKEN'])
print('Dán vào AIFlow → Kiểm tra → Lưu và tiếp tục. Không chia sẻ output này.')

## B. Batch không cần tunnel

Trong AIFlow, mở **Dùng batch khi không có tunnel**, dán JSON profile từ cell nạp model rồi lưu. Tạo tác vụ và tải **Xuất batch JSON**. Upload JSON vào cell dưới, chờ tạo xong rồi nhập ZIP về đúng tác vụ AIFlow. Có thể chạy lại JSON cũ để dùng checkpoint. ZIP giới hạn 128 MiB ở bản đầu, nên chia bài dài thành nhiều tác vụ.

In [ ]:
batch_upload = files.upload()
if len(batch_upload) != 1:
    raise ValueError('Chọn một manifest JSON từ AIFlow')
manifest = json.loads(next(iter(batch_upload.values())))
result_zip = Path('/content/aiflow-audio-results.zip')
worker.run_batch(manifest, result_zip)
files.download(str(result_zip))

## Kết thúc phiên

Cell dưới chỉ dừng API/tunnel. Sau đó chọn **Runtime → Disconnect and delete runtime** để giải phóng GPU. Đóng tab không thay thế bước này. File audio và checkpoint vẫn nằm trong thư mục Drive riêng của bạn.

In [ ]:
if worker.ACTIVE:
    raise RuntimeError('Còn tác vụ đang chạy. Chờ hoàn thành hoặc hủy trong AIFlow trước.')
if 'tunnel' in globals() and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=10)
if 'server' in globals():
    server.should_exit = True
if 'api_thread' in globals():
    api_thread.join(timeout=10)
print('Đã yêu cầu dừng API. Bây giờ chọn Runtime → Disconnect and delete runtime.')